In [1]:
import duckdb as dd

In [2]:
# Create a persistent connection to duckdb to save data for use later. Extracted folder should be labelled the same, otherwise file path can be changed to what contains the json objects.
conn = dd.connect('spotify_all_years.db')

In [ ]:
#First table: combine all year's JSON files into one
conn.execute(
"CREATE TABLE complete_data AS " \
"SELECT * " \
"FROM read_json_auto('Spotify Extended Streaming History/*.json', union_by_name=true)"
)

In [ ]:
#Inspect data
conn.execute("SELECT * FROM complete_data LIMIT 5").df()

,ts,platform,ms_played,conn_country,ip_addr,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,episode_name,...,audiobook_uri,audiobook_chapter_uri,audiobook_chapter_title,reason_start,reason_end,shuffle,skipped,offline,offline_timestamp,incognito_mode
0,2016-12-07 22:36:35,"Android-tablet OS 4.4.2 API 19 (Gigabyte, RCT6...",144244,US,97.71.20.155,Juju on That Beat (TZ Anthem),Zay Hilfigerrr,Juju on That Beat (TZ Anthem),spotify:track:1lItf5ZXJc1by9SbPeljFd,None,...,None,None,None,trackdone,trackdone,True,False,False,<NA>,False
1,2016-12-07 22:37:28,"Android-tablet OS 4.4.2 API 19 (Gigabyte, RCT6...",50521,US,97.71.20.155,"i hate u, i love u (feat. olivia o'brien)",gnash,us,spotify:track:7vRriwrloYVaoAe3a9wJHe,None,...,None,None,None,trackdone,fwdbtn,True,False,False,<NA>,False
2,2016-12-07 22:41:17,"Android-tablet OS 4.4.2 API 19 (Gigabyte, RCT6...",230453,US,97.71.20.155,Starboy,The Weeknd,Starboy,spotify:track:7MXVkk9YMctZqd1Srtv4MB,None,...,None,None,None,fwdbtn,trackdone,True,False,False,<NA>,False
3,2016-12-07 22:54:06,"Android-tablet OS 4.4.2 API 19 (Gigabyte, RCT6...",214480,US,97.71.20.155,Don't Wanna Know,Maroon 5,Don't Wanna Know,spotify:track:0AKejOKlGdiB53QpwAeenO,None,...,None,None,None,trackdone,trackdone,True,False,False,<NA>,False
4,2016-12-07 22:54:13,"Android-tablet OS 4.4.2 API 19 (Gigabyte, RCT6...",5022,US,97.71.20.155,One Call Away,Charlie Puth,Nine Track Mind,spotify:track:7soJgKhQTO8hLP2JPRkL5O,None,...,None,None,None,trackdone,fwdbtn,True,False,False,<NA>,False


In [ ]:
# Check how many columns + non-null values 
conn.execute("SELECT * FROM complete_data").df().info()

<class 'pandas.DataFrame'>
RangeIndex: 101657 entries, 0 to 101656
Data columns (total 23 columns):
 #   Column                             Non-Null Count   Dtype         
---  ------                             --------------   -----         
 0   ts                                 101657 non-null  datetime64[us]
 1   platform                           101657 non-null  str           
 2   ms_played                          101657 non-null  int64         
 3   conn_country                       101657 non-null  str           
 4   ip_addr                            101657 non-null  str           
 5   master_metadata_track_name         100994 non-null  str           
 6   master_metadata_album_artist_name  100994 non-null  str           
 7   master_metadata_album_album_name   100994 non-null  str           
 8   spotify_track_uri                  100994 non-null  str           
 9   episode_name                       597 non-null     str           
 10  episode_show_name              

In [ ]:
#remove columns we won't use and make a new table
conn.execute("CREATE TABLE clean_data AS SELECT ts,platform, ms_played, ip_addr,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,reason_start,reason_end,shuffle,skipped FROM complete_data")

In [ ]:
#verify columns were dropped
conn.execute("SELECT * FROM clean_data").df().info()

<class 'pandas.DataFrame'>
RangeIndex: 101657 entries, 0 to 101656
Data columns (total 12 columns):
 #   Column                             Non-Null Count   Dtype         
---  ------                             --------------   -----         
 0   ts                                 101657 non-null  datetime64[us]
 1   platform                           101657 non-null  str           
 2   ms_played                          101657 non-null  int64         
 3   ip_addr                            101657 non-null  str           
 4   master_metadata_track_name         100994 non-null  str           
 5   master_metadata_album_artist_name  100994 non-null  str           
 6   master_metadata_album_album_name   100994 non-null  str           
 7   spotify_track_uri                  100994 non-null  str           
 8   reason_start                       101657 non-null  str           
 9   reason_end                         101657 non-null  str           
 10  shuffle                        

In [11]:
# dicrepancy between # entries and tracks because podcasts are still counted.
# remove podcasts/episodes from entries, Ran against just one column for optimization.
conn.execute("CREATE TABLE podcast_dropped AS SELECT * FROM clean_data WHERE master_metadata_album_artist_name IS NOT NULL")
#conn.execute("CREATE TABLE podcast_dropped AS SELECT * FROM clean_data WHERE master_metadata_track_name IS NOT NULL AND master_metadata_album_artist_name IS NOT NULL AND master_metadata_album_album_name IS NOT NULL AND spotify_track_uri IS NOT NULL")
#Another way to perform the same query but would alter original table:
#conn.execute("DELETE FROM clean_data WHERE master_metadata_track_name IS NULL AND master_metadata_album_artist_name IS NULL AND master_metadata_album_album_name IS NULL AND spotify_track_uri IS NULL ")

In [3]:
#verify # of entries went down (meaning podcasts and episodes removed)
conn.execute("SELECT * FROM podcast_dropped").df().info()

<class 'pandas.DataFrame'>
RangeIndex: 100994 entries, 0 to 100993
Data columns (total 12 columns):
 #   Column                             Non-Null Count   Dtype         
---  ------                             --------------   -----         
 0   ts                                 100994 non-null  datetime64[us]
 1   platform                           100994 non-null  str           
 2   ms_played                          100994 non-null  int64         
 3   ip_addr                            100994 non-null  str           
 4   master_metadata_track_name         100994 non-null  str           
 5   master_metadata_album_artist_name  100994 non-null  str           
 6   master_metadata_album_album_name   100994 non-null  str           
 7   spotify_track_uri                  100994 non-null  str           
 8   reason_start                       100994 non-null  str           
 9   reason_end                         100994 non-null  str           
 10  shuffle                        

In [12]:
# num of diff songs and diff albums listened to by artist.
# NOT accurate includes released singles as albums.
conn.execute("SELECT master_metadata_album_artist_name AS artist_name, COUNT(DISTINCT master_metadata_album_album_name) AS num_albums_per_artist, COUNT(DISTINCT master_metadata_track_name) AS num_songs_per_artist FROM podcast_dropped GROUP BY artist_name").df()

,artist_name,num_albums_per_artist,num_songs_per_artist
0,Jonas Blue,3,4
1,Palmer,1,1
2,awfultune,3,3
3,Meek Mill,2,2
4,The Ivy,3,2
...,...,...,...
2728,SAZZA,1,1
2729,Parlour Steps,1,1
2730,The Memories,1,1
2731,LiLucifer,1,1


In [ ]:
# distinct tracks per album per artist
# Limitation: if a user listened to an artist but only 1 song of an album it will not be included because it is considering it a single
conn.execute("SELECT master_metadata_album_artist_name AS artist_name, master_metadata_album_album_name AS album_name, COUNT(DISTINCT master_metadata_track_name) AS num_tracks " \
"FROM podcast_dropped " \
"GROUP BY artist_name, album_name " \
"HAVING num_tracks > 1").df()

,artist_name,album_name,num_tracks
0,Joji,In Tongues (Deluxe),5
1,blackbear,deadroses,2
2,Ellie Goulding,Delirium,3
3,Katy Perry,PRISM,5
4,Juice WRLD,Death Race For Love - Bonus Track Version,8
...,...,...,...
973,NLE Choppa,SLUT SZN,5
974,Cavetown,Animal Kingdom,2
975,Kanye West,Donda,3
976,Zach Bryan,Zach Bryan,2


In [ ]:
#One row per artist, with a count of only the albums where you listened to more than 1 track. Non-single albumbs per artist.
conn.execute("SELECT artist_name, COUNT(album_name) AS num_true_albums" \
" FROM (SELECT master_metadata_album_artist_name AS artist_name, master_metadata_album_album_name AS album_name, COUNT(DISTINCT master_metadata_track_name) AS num_tracks FROM podcast_dropped GROUP BY artist_name, album_name HAVING num_tracks > 1) AS album_counts" \
" GROUP BY artist_name").df()

,artist_name,num_true_albums
0,Bhad Bhabie,1
1,ROLE MODEL,4
2,Addison Rae,2
3,H.E.R.,1
4,Hailee Steinfeld,2
...,...,...
482,jxdn,1
483,Fred again..,1
484,50 Cent,2
485,Amy Winehouse,1


In [3]:
# Total play count per artist
conn.execute("SELECT COUNT(*) AS play_count_per_artist,master_metadata_album_artist_name AS artist_name"\
" FROM podcast_dropped" \
" GROUP BY master_metadata_album_artist_name").df()

,play_count_per_artist,artist_name
0,252,Hailee Steinfeld
1,50,The Vamps
2,8,Jonas Blue
3,5,Marc E. Bassy
4,198,Calvin Harris
...,...,...
2728,2,The Two Lips
2729,1,Dustin O'Halloran
2730,1,3ee
2731,1,Dutch Melrose


In [ ]:
# Listening consistency — COUNT(DISTINCT month/year) per artist. Strongest superfan signal you have.
# artist_name | months_active (Jan2019, Feb2019, March2020 -> counted up)
# wallows     | 18
conn.execute("SELECT master_metadata_album_artist_name, COUNT(DISTINCT strftime('%Y-%m', ts)) AS listening_consistency " \
" FROM podcast_dropped" \
" GROUP BY master_metadata_album_artist_name").df()

,master_metadata_album_artist_name,listening_consistency
0,Clemency,1
1,Miley Cyrus,27
2,Kelsea Ballerini,1
3,Fatboibari,21
4,5 Seconds of Summer,39
...,...,...
2728,astra division,1
2729,Part Time,1
2730,The Juliets,1
2731,Skye,1


In [ ]:
# Skip rate per artist, higher = you skip the artist most of the time, lower = you let them play out.
conn.execute("SELECT master_metadata_album_artist_name, AVG(CAST(skipped AS INT)) as skip_rate" \
" FROM podcast_dropped" \
" GROUP BY master_metadata_album_artist_name").df()


,master_metadata_album_artist_name,skip_rate
0,Hailee Steinfeld,0.031746
1,The Vamps,0.020000
2,Jonas Blue,0.125000
3,Marc E. Bassy,0.200000
4,Calvin Harris,0.186869
...,...,...
2728,Henry Morris,1.000000
2729,Dexter and The Moonrocks,1.000000
2730,Papa Roach,1.000000
2731,vain,1.000000


In [66]:
# Intentional listening rate: percentage of plays where reason_start IN ('clickrow', 'playbtn', 'backbtn') per artist.
conn.execute("SELECT master_metadata_album_artist_name, SUM(CAST(reason_start IN ('clickrow', 'playbtn', 'backbtn') AS INT)) / COUNT(*) AS intentional_listening_rate" \
" FROM podcast_dropped" \
" GROUP BY master_metadata_album_artist_name").df()

,master_metadata_album_artist_name,intentional_listening_rate
0,Troye Sivan,0.208511
1,Rae Sremmurd,0.205607
2,Young M.A,0.000000
3,WILDES,1.000000
4,Betty Who,1.000000
...,...,...
2728,Chevelle,0.000000
2729,Silverchair,0.000000
2730,John Wells,1.000000
2731,Da LAB,1.000000


In [75]:
# Completion rate — AVG(ms_played) per artist as a proxy (you don't have track duration, so average ms played is the best you can do).
conn.execute("SELECT master_metadata_album_artist_name, SUM(CAST(reason_end = 'trackdone' AS INT)) / COUNT(*) AS completion_rate" \
" FROM podcast_dropped" \
" GROUP BY master_metadata_album_artist_name").df() 

,master_metadata_album_artist_name,completion_rate
0,Hailee Steinfeld,0.583333
1,The Vamps,0.660000
2,Jonas Blue,0.000000
3,Marc E. Bassy,0.000000
4,Calvin Harris,0.555556
...,...,...
2728,Chevelle,0.000000
2729,Silverchair,0.000000
2730,John Wells,0.000000
2731,Da LAB,0.000000
